# Option B — Data Acquisition Discovery Notebook

**Purpose:** Explore and acquire raw data for Option B.
Raw data -> `../../product/data/raw/` | Cleaned data -> `../modified_data/`

**Sources**
1. CMS Provider Data Catalog (DKAN metastore) — ED timeliness OP-18b
2. US Census ACS 5-Year 2022 — state poverty, income, uninsured, population
3. NPI Registry — primary care physician counts by state

> **API Change:** CMS migrated from Socrata to DKAN.
> Old: `data.cms.gov/resource/yv7e-xc69.json` (deprecated)
> New: metastore `schemas/dataset/items/yv7e-xc69` → datastore query or direct CSV download.

In [ ]:
import json
import time
import pathlib
import requests
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()  # loads CENSUS_API_KEY from repo-root .env

RAW = pathlib.Path("../../product/data/raw")
MOD = pathlib.Path("../modified_data")
RAW.mkdir(parents=True, exist_ok=True)
MOD.mkdir(parents=True, exist_ok=True)

CENSUS_KEY = os.getenv("CENSUS_API_KEY", "")
print("Raw dir :", RAW.resolve())
print("Census key present:", bool(CENSUS_KEY))

---
## 1. CMS Timely and Effective Care (DKAN Metastore → Datastore)

### Step 1a — Discover dataset metadata and distribution IDs

In [ ]:
DATASET_ID = "yv7e-xc69"  # Timely and Effective Care - Hospital
META_URL = (
    "https://data.cms.gov/provider-data/api/1/metastore"
    f"/schemas/dataset/items/{DATASET_ID}"
)

meta = requests.get(META_URL, params={"show-reference-ids": "true"}, timeout=30).json()

# Persist full metadata snapshot for reproducibility
(RAW / f"cms_ed_meta_{DATASET_ID}.json").write_text(json.dumps(meta, indent=2))

print("Title    :", meta.get("title"))
print("Modified :", meta.get("modified"))
print("# distrib:", len(meta.get("distribution", [])))

In [ ]:
# Inspect all distributions — find CSV download URL and/or datastore resource IDs
distributions = meta.get("distribution", [])
for i, d in enumerate(distributions):
    dat = d.get("data", {})
    print(f"[{i}] id={d.get('identifier')}")
    print(f"     mediaType  = {dat.get('mediaType')}")
    print(f"     downloadURL= {dat.get('downloadURL', 'N/A')[:100]}")
    print()

### Step 1b — Direct CSV download (preferred: no pagination needed)

In [ ]:
csv_dist = next(
    (d for d in distributions
     if "csv" in d.get("data", {}).get("mediaType", "").lower()),
    None
)

if csv_dist:
    csv_url = csv_dist["data"]["downloadURL"]
    print("CSV URL:", csv_url[:120])
    df_full = pd.read_csv(csv_url)
    print("Shape  :", df_full.shape)
    print("Columns:", df_full.columns.tolist())
else:
    print("No CSV distribution found — use Step 1c (datastore API)")

In [ ]:
# Identify column names (may vary between CMS data releases)
measure_col = next((c for c in df_full.columns if "measure" in c.lower() and "id" in c.lower()), None)
score_col   = next((c for c in df_full.columns if "score"   in c.lower()), None)
state_col   = next((c for c in df_full.columns if c.lower() == "state"), None)

print("measure_col:", measure_col)
print("score_col  :", score_col)
print("state_col  :", state_col)
print()
print("Top measure IDs:")
print(df_full[measure_col].value_counts().head(15))

In [ ]:
# Filter to OP-18b — adjust TARGET_MEASURE if column uses different casing
TARGET_MEASURE = "OP_18b"
df_op18b = df_full[df_full[measure_col].str.strip().str.upper() == TARGET_MEASURE.upper()].copy()
df_op18b[score_col] = pd.to_numeric(df_op18b[score_col], errors="coerce")

print(f"Rows for {TARGET_MEASURE}: {len(df_op18b)}")
print(f"Score range: {df_op18b[score_col].min():.0f} – {df_op18b[score_col].max():.0f} min")
df_op18b.head(3)

In [ ]:
out = RAW / "cms_ed_timeliness_op18b.csv"
df_op18b.to_csv(out, index=False)
print("Saved:", out)

### Step 1c — Datastore query API (alternative: filtered + paginated)

Use when no direct CSV distribution is available, or when the full CSV is very large.

In [ ]:
# Use the first distribution's identifier as the resource ID
# Adjust the index based on Step 1a output
resource_id   = distributions[0]["identifier"]
DATASTORE_URL = f"https://data.cms.gov/provider-data/api/1/datastore/query/{resource_id}/0"
print("Resource ID   :", resource_id)
print("Datastore URL :", DATASTORE_URL)

# Probe with 5 rows
probe = requests.post(DATASTORE_URL, json={
    "conditions": [{"property": "measure_id", "value": "OP_18b", "operator": "="}],
    "limit": 5, "offset": 0,
}, timeout=30).json()

print("Response keys :", list(probe.keys()))
print("Total count   :", probe.get("count", "N/A"))
if probe.get("results"):
    print("Row fields    :", list(probe["results"][0].keys()))

In [ ]:
# Full paginated pull via DKAN datastore
all_rows = []
limit    = 500
offset   = 0

while True:
    resp = requests.post(DATASTORE_URL, json={
        "conditions": [{"property": "measure_id", "value": "OP_18b", "operator": "="}],
        "limit":  limit,
        "offset": offset,
        "sort":   [{"property": "facility_id", "order": "asc"}],
    }, timeout=30)
    resp.raise_for_status()
    rows = resp.json().get("results", [])
    if not rows:
        break
    all_rows.extend(rows)
    print(f"  fetched {len(all_rows)} rows (offset={offset})")
    if len(rows) < limit:
        break
    offset += limit
    time.sleep(0.2)

print(f"Total: {len(all_rows)} rows")
(RAW / "cms_ed_timeliness_op18b_datastore.json").write_text(json.dumps(all_rows, indent=2))
print("Saved raw JSON.")

---
## 2. US Census ACS 5-Year 2022 — State-Level Covariates

In [ ]:
ACS_BASE = "https://api.census.gov/data/2022/acs/acs5"
acs_vars = ",".join(["NAME", "B17001_002E", "B17001_001E", "B01003_001E", "B19013_001E"])
# B17001_002E = persons below poverty  | B17001_001E = poverty universe
# B01003_001E = total population       | B19013_001E = median HH income

raw_acs = requests.get(
    ACS_BASE,
    params={"get": acs_vars, "for": "state:*", "key": CENSUS_KEY},
    timeout=30
).json()

(RAW / "acs_state_2022.json").write_text(json.dumps(raw_acs, indent=2))
df_acs = pd.DataFrame(raw_acs[1:], columns=raw_acs[0])
print("ACS shape:", df_acs.shape)
df_acs.head(3)

In [ ]:
# Uninsured rate from ACS subject tables
# S2701_C04_001E = % uninsured, civilian noninstitutionalized population
raw_unins = requests.get(
    "https://api.census.gov/data/2022/acs/acs5/subject",
    params={"get": "NAME,S2701_C04_001E", "for": "state:*", "key": CENSUS_KEY},
    timeout=30
).json()

(RAW / "acs_uninsured_state_2022.json").write_text(json.dumps(raw_unins, indent=2))
df_unins = pd.DataFrame(raw_unins[1:], columns=raw_unins[0])
df_unins = df_unins.rename(columns={"S2701_C04_001E": "uninsured_pct"})
df_unins["uninsured_pct"] = pd.to_numeric(df_unins["uninsured_pct"], errors="coerce")
print("Uninsured shape:", df_unins.shape)

In [ ]:
num_cols = ["B17001_002E", "B17001_001E", "B01003_001E", "B19013_001E"]
df_acs[num_cols] = df_acs[num_cols].apply(pd.to_numeric, errors="coerce")
df_acs["poverty_rate"] = df_acs["B17001_002E"] / df_acs["B17001_001E"] * 100
df_acs = df_acs.rename(columns={"B01003_001E": "population", "B19013_001E": "median_income"})

df_acs_full = df_acs.merge(df_unins[["state", "uninsured_pct"]], on="state", how="left")
df_acs_full[["NAME", "poverty_rate", "population", "median_income", "uninsured_pct"]].head(5)

---
## 3. NPI Registry — Primary Care Physician Counts by State

> **Note:** Full pull (51 states × 4 taxonomies) takes ~20–40 min.
> Each (state, taxonomy) batch is cached as a JSON file; re-runs are instant.

In [ ]:
NPI_URL = "https://npiregistry.cms.hhs.gov/api/"

# Quick probe: CA / Family Medicine
probe_npi = requests.get(NPI_URL, params={
    "version": "2.1", "enumeration_type": "NPI-1",
    "taxonomy_description": "Family Medicine",
    "state": "CA", "limit": 5, "skip": 0,
}, timeout=30).json()

print("Result count CA/Family Medicine:", probe_npi.get("result_count"))
if probe_npi.get("results"):
    print("Sample NPI:", probe_npi["results"][0].get("number"))

In [ ]:
STATES = [
    "AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID","IL","IN",
    "IA","KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH",
    "NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT",
    "VT","VA","WA","WV","WI","WY"
]
TAXONOMIES = ["Family Medicine", "Internal Medicine", "General Practice", "Pediatrics"]

NPI_RAW_DIR = RAW / "npi_pcp"
NPI_RAW_DIR.mkdir(exist_ok=True)
npi_summary = {}  # state -> unique PCP count

for state in STATES:
    state_npis = set()
    for taxonomy in TAXONOMIES:
        cache = NPI_RAW_DIR / f"npi_{state}_{taxonomy.replace(' ', '_')}.json"
        if cache.exists():
            records = json.loads(cache.read_text())
        else:
            records, skip = [], 0
            while True:
                r = requests.get(NPI_URL, params={
                    "version": "2.1", "enumeration_type": "NPI-1",
                    "taxonomy_description": taxonomy,
                    "state": state, "limit": 200, "skip": skip,
                }, timeout=30)
                if r.status_code != 200:
                    print(f"  WARN {state}/{taxonomy}: HTTP {r.status_code}")
                    break
                batch = r.json().get("results", [])
                if not batch:
                    break
                records.extend(batch)
                if len(batch) < 200:
                    break
                skip += 200
                time.sleep(0.15)
            cache.write_text(json.dumps(records))
        state_npis.update(rec["number"] for rec in records if rec.get("number"))
    npi_summary[state] = len(state_npis)
    print(f"{state}: {len(state_npis)} unique PCPs")

(RAW / "npi_pcp_state_counts.json").write_text(json.dumps(npi_summary, indent=2))
print("Done — NPI summary saved.")

---
## 4. Preliminary Join — State Analysis Table

Saves `state_analysis_table.csv` to `modified_data/` for EDA notebooks.

In [ ]:
STATE_NAME_TO_ABBR = {
    "Alabama":"AL","Alaska":"AK","Arizona":"AZ","Arkansas":"AR","California":"CA",
    "Colorado":"CO","Connecticut":"CT","Delaware":"DE","District of Columbia":"DC",
    "Florida":"FL","Georgia":"GA","Hawaii":"HI","Idaho":"ID","Illinois":"IL",
    "Indiana":"IN","Iowa":"IA","Kansas":"KS","Kentucky":"KY","Louisiana":"LA",
    "Maine":"ME","Maryland":"MD","Massachusetts":"MA","Michigan":"MI","Minnesota":"MN",
    "Mississippi":"MS","Missouri":"MO","Montana":"MT","Nebraska":"NE","Nevada":"NV",
    "New Hampshire":"NH","New Jersey":"NJ","New Mexico":"NM","New York":"NY",
    "North Carolina":"NC","North Dakota":"ND","Ohio":"OH","Oklahoma":"OK","Oregon":"OR",
    "Pennsylvania":"PA","Rhode Island":"RI","South Carolina":"SC","South Dakota":"SD",
    "Tennessee":"TN","Texas":"TX","Utah":"UT","Vermont":"VT","Virginia":"VA",
    "Washington":"WA","West Virginia":"WV","Wisconsin":"WI","Wyoming":"WY",
}
df_acs_full["state_abbr"] = df_acs_full["NAME"].map(STATE_NAME_TO_ABBR)

df_npi = pd.DataFrame(list(npi_summary.items()), columns=["state_abbr", "pcp_count"])
df_merged = df_acs_full.merge(df_npi, on="state_abbr", how="left")
df_merged["pcp_per_100k"] = df_merged["pcp_count"] / (df_merged["population"] / 100_000)
print("Merged shape:", df_merged.shape)

In [ ]:
# Aggregate CMS hospitals to state median ED wait
df_cms = pd.read_csv(RAW / "cms_ed_timeliness_op18b.csv")
score_col = next((c for c in df_cms.columns if "score" in c.lower()), "Score")
state_col = next((c for c in df_cms.columns if c.lower() == "state"),  "State")
df_cms[score_col] = pd.to_numeric(df_cms[score_col], errors="coerce")

df_state_ed = (
    df_cms.groupby(state_col)
    .agg(ed_wait_median=(score_col, "median"), n_hospitals=(score_col, "count"))
    .reset_index()
    .rename(columns={state_col: "state_abbr"})
)
print("State ED rows:", len(df_state_ed))
df_state_ed.sort_values("ed_wait_median", ascending=False).head(10)

In [ ]:
df_analysis = df_state_ed.merge(df_merged, on="state_abbr", how="left")
print("Analysis table shape:", df_analysis.shape)
print("Null ED wait  :", df_analysis["ed_wait_median"].isna().sum())
print("Null pcp/100k :", df_analysis["pcp_per_100k"].isna().sum())

out = MOD / "state_analysis_table.csv"
df_analysis.to_csv(out, index=False)
print("Saved:", out)
df_analysis[["state_abbr", "ed_wait_median", "poverty_rate", "pcp_per_100k", "uninsured_pct"]].head(10)